# Sommelier Kaggle Full Trace Run

Notebook này chạy lại pipeline từ đầu và lưu output sau từng bước để dễ kiểm tra.

Điều kiện trước khi chạy:
- Kaggle Accelerator: GPU bật.
- Kaggle Internet: bật.
- Kaggle Secret có `HF_TOKEN`.
- Dataset audio đã add vào notebook. Notebook sẽ tự tìm file audio đầu tiên trong `/kaggle/input`.

Lưu ý: code hiện tại của repo chạy thứ tự `stage_01_diarize -> stage_02_music_clean -> stage_03_overlap_separate -> stage_04_asr -> stage_05_export`. Nếu muốn đảo `overlap separation` lên trước `music clean` giống diagram gốc thì phải sửa code pipeline, không chỉ đổi lệnh notebook.

## 0. Cấu hình run

Chỉnh các biến bên dưới nếu muốn đổi branch, giới hạn thời lượng test, hoặc tắt bước nặng.

In [ ]:
REPO_URL = "https://github.com/lamkdhe180931-arch/sommelier.git"
BRANCH = "test-divide-stage"

RUN_DIR = "/kaggle/working/run_full"
AUDIO_WAV = "/kaggle/working/audio/full.wav"

# Để None nếu muốn chạy full audio. Để 300 nếu muốn test nhanh 5 phút.
AUDIO_LIMIT_SECONDS = None

# Bước nặng. Có thể tắt để debug nhanh.
RUN_DEMUCS = True
RUN_SEPREFORMER = True

# ASRMoE chạy cả 3 model tiếng Việt: Whisper + PhoWhisper + ChunkFormer.
# Trên Kaggle 2xT4: Whisper đặt GPU0, PhoWhisper/ChunkFormer đặt GPU1.
ASR_MOE = True
WHISPER_DEVICE_INDEX = 0
VI_ASR_DEVICE_INDEX = 1
WHISPER_ARCH = "large-v3"
COMPUTE_TYPE = "float16"
ASR_THREADS = 4

HF_SECRET_NAME = "HF_TOKEN"

## 1. Clone repo

In [ ]:
%cd /kaggle/working
!rm -rf /kaggle/working/sommelier
!git clone -b "{BRANCH}" "{REPO_URL}" /kaggle/working/sommelier
%cd /kaggle/working/sommelier/podcast-pipeline
!pwd
!git rev-parse --abbrev-ref HEAD
!git log -1 --oneline

## 2. Cài dependencies

Cell này mất thời gian. Sau khi cài xong, notebook sẽ pin lại `numpy==2.2.6`, `numba==0.61.2`, `llvmlite==0.44.0` để tránh lỗi `Numba needs NumPy 2.2 or less`.

In [ ]:
%%time
%cd /kaggle/working/sommelier/podcast-pipeline

!apt-get update -y
!apt-get install -y ffmpeg git git-lfs

!python -m pip install -U pip setuptools wheel packaging ninja

# Cài requirements repo, bỏ nemo-toolkit[all]
!grep -v "nemo-toolkit\\[all\\]" requirements.txt > requirements-kaggle.txt
!python -m pip install -r requirements-kaggle.txt

# Cài NeMo ASR cho Sortformer diarization.
!python -m pip uninstall -y nemo-toolkit lightning pytorch-lightning
!python -m pip install "lightning==2.4.0" "pytorch-lightning==2.5.2"
!python -m pip install "nemo-toolkit[asr]==2.4.0"

# Quan trọng: cài lại torch stack cuối cùng để sửa torchvision::nms
!python -m pip uninstall -y torch torchvision torchaudio
!python -m pip install --no-cache-dir --force-reinstall \
  torch==2.7.1 torchaudio==2.7.1 torchvision==0.22.1 \
  --index-url https://download.pytorch.org/whl/cu126

# --- CHỐT CHẶN LỖI PILLOW (_Ink) ---
!python -m pip install "pillow<12.0"

# Pin torchmetrics, không cho nó kéo lại torch/torchvision
!python -m pip install --no-cache-dir --force-reinstall --no-deps "torchmetrics==1.7.4"

# Pin numpy/numba cuối cùng
!python -m pip install --no-cache-dir --force-reinstall \
  "numpy==2.2.6" "numba==0.61.2" "llvmlite==0.44.0"

## 3. Kiểm tra môi trường

In [ ]:
import importlib.metadata as importlib_metadata
import numpy, numba, torch
print("nemo-toolkit:", importlib_metadata.version("nemo-toolkit"))
print("chunkformer:", importlib_metadata.version("chunkformer"))
print("numpy:", numpy.__version__)
print("numba:", numba.__version__)
print("torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

import whisperx
print("whisperx ok")

import nemo.collections.asr as nemo_asr
print("nemo asr ok")

from nemo.collections.asr.models import SortformerEncLabelModel
print("sortformer import ok")

## 4. Gắn Hugging Face token vào config

In [ ]:
import json
from kaggle_secrets import UserSecretsClient
from huggingface_hub import whoami

token = UserSecretsClient().get_secret(HF_SECRET_NAME)
print("HF token:", token[:8] + "..." if token else "missing")
print(whoami(token=token))

with open("config.json", "r", encoding="utf-8") as f:
    cfg = json.load(f)

cfg["huggingface_token"] = token

with open("config.json", "w", encoding="utf-8") as f:
    json.dump(cfg, f, indent=2, ensure_ascii=False)

print("config.json updated")

## 5. Tìm audio input và chuẩn hóa audio

Output: `AUDIO_WAV = /kaggle/working/audio/full.wav`.

Chuẩn hóa về mono 16 kHz để các model dùng cùng format.

In [ ]:
from pathlib import Path

audio_exts = {".mp3", ".wav", ".m4a", ".flac", ".aac", ".ogg"}
audio_candidates = sorted(
    p for p in Path("/kaggle/input").rglob("*")
    if p.is_file() and p.suffix.lower() in audio_exts
)

if not audio_candidates:
    raise FileNotFoundError("Không tìm thấy audio trong /kaggle/input. Hãy Add Input hoặc Upload audio trước.")

AUDIO_IN = str(audio_candidates[0])
print("AUDIO_IN:", AUDIO_IN)
print("AUDIO_WAV:", AUDIO_WAV)
print("RUN_DIR:", RUN_DIR)

!mkdir -p /kaggle/working/audio "{RUN_DIR}"

limit_arg = f"-t {AUDIO_LIMIT_SECONDS}" if AUDIO_LIMIT_SECONDS else ""
!ffmpeg -hide_banner -y -i "{AUDIO_IN}" {limit_arg} -ac 1 -ar 16000 "{AUDIO_WAV}"

In [ ]:
from pydub import AudioSegment
from IPython.display import Audio, display

audio = AudioSegment.from_file(AUDIO_WAV)
print("Audio:", AUDIO_WAV)
print("Duration seconds:", len(audio) / 1000)
print("Frame rate:", audio.frame_rate)
print("Channels:", audio.channels)
display(Audio(AUDIO_WAV))

## 6. Tải model phụ cho music clean và overlap separation

In [ ]:
from huggingface_hub import hf_hub_download

if RUN_DEMUCS:
    panns_path = hf_hub_download(
        repo_id="thelou1s/panns-inference",
        filename="Cnn14_mAP=0.431.pth",
        local_dir="/kaggle/working/sommelier/panns_data",
    )
    print("PANNs checkpoint:", panns_path)
else:
    print("RUN_DEMUCS=False, bỏ qua tải PANNs")

In [ ]:
import os
import subprocess
from pathlib import Path

if RUN_SEPREFORMER:
    subprocess.run("git lfs install", shell=True, check=True)
    os.chdir("/kaggle/working/sommelier")
    if not Path("SepReformer").exists():
        subprocess.run("git clone https://github.com/dmlguq456/SepReformer.git SepReformer", shell=True, check=True)
    os.chdir("/kaggle/working/sommelier/SepReformer")
    subprocess.run("git lfs pull", shell=True, check=True)
    subprocess.run("python -m pip install --no-deps mir-eval==0.7 ptflops==0.7.4 thop==0.1.1.post2209072238 torchinfo==1.8.0", shell=True, check=True)

    log = Path("/kaggle/working/sommelier/SepReformer/models/SepReformer_Base_WSJ0/log")
    src = log / "scratch_weight"
    dst = log / "scratch_weights"
    if src.exists() and not dst.exists():
        os.symlink(src, dst)

    ckpts = list(log.rglob("*.pt")) + list(log.rglob("*.pth"))
    print("SepReformer checkpoints:")
    for p in ckpts[:20]:
        print(p)
else:
    print("RUN_SEPREFORMER=False, bỏ qua SepReformer")

os.chdir("/kaggle/working/sommelier/podcast-pipeline")

## 7. Trace VAD chunking

Bước này chỉ để xem VAD chia audio thành các chunk dài thế nào trước diarization. Đây không phải output speaker segment cuối cùng.

Output:
- `/kaggle/working/run_full/trace_vad_chunks.json`
- `/kaggle/working/run_full/vad_chunks/*.wav`

In [ ]:
%cd /kaggle/working/sommelier/podcast-pipeline

import json
import shutil
from pathlib import Path
import pandas as pd
from pydub import AudioSegment
import stage_common
import main_original_ASR_MoE as pipeline

cfg = pipeline.load_cfg("config.json")
logger = pipeline.Logger.get_logger()
pipeline.cfg = cfg
pipeline.logger = logger

device_name = "cuda" if pipeline.torch.cuda.is_available() else "cpu"
device = pipeline.torch.device(device_name)
pipeline.device_name = device_name
pipeline.device = device
pipeline.vad = pipeline.silero_vad.SileroVAD(device=device)

sample_rate = int(cfg["entrypoint"]["SAMPLE_RATE"])
audio_info = stage_common.load_audio_info(AUDIO_WAV, sample_rate)
diar_chunks, temp_chunk_dir = pipeline.prepare_diarization_chunks(AUDIO_WAV, audio_info)

chunk_dir = Path(RUN_DIR) / "vad_chunks"
chunk_dir.mkdir(parents=True, exist_ok=True)

trace_chunks = []
for idx, chunk in enumerate(diar_chunks):
    src = Path(chunk["path"])
    dst = chunk_dir / f"chunk_{idx:03d}.wav"
    shutil.copy2(src, dst)
    duration = AudioSegment.from_file(dst).duration_seconds
    trace_chunks.append({
        "index": f"{idx:03d}",
        "path": str(dst),
        "offset": float(chunk["offset"]),
        "duration": float(duration),
        "start": float(chunk["offset"]),
        "end": float(chunk["offset"] + duration),
    })

if temp_chunk_dir:
    shutil.rmtree(temp_chunk_dir, ignore_errors=True)

stage_common.dump_json({
    "audio_path": AUDIO_WAV,
    "sample_rate": sample_rate,
    "chunks": trace_chunks,
    "metadata": {"stage": "vad_chunk_trace"},
}, Path(RUN_DIR) / "trace_vad_chunks.json")

df_chunks = pd.DataFrame(trace_chunks)
print("VAD chunks:", len(df_chunks))
display(df_chunks.head(20))

In [ ]:
from IPython.display import Audio, display

if trace_chunks:
    print(trace_chunks[0])
    display(Audio(trace_chunks[0]["path"]))

## 8. Stage 01 - Speaker diarization

Output: `/kaggle/working/run_full/diarization.json`.

Đây là bước Sortformer + speaker linking, tạo segment có `start`, `end`, `speaker`.

In [ ]:
%cd /kaggle/working/sommelier/podcast-pipeline

!python stage_01_diarize.py \
  --input_audio "{AUDIO_WAV}" \
  --out "{RUN_DIR}/diarization.json" \
  --merge_gap 2.0 \
  --max_segment_duration 30.0

In [ ]:
import json
import pandas as pd
from IPython.display import display

with open(f"{RUN_DIR}/diarization.json", "r", encoding="utf-8") as f:
    diar = json.load(f)

diar_segments = diar["segments"]
df_diar = pd.DataFrame(diar_segments)
df_diar["dur"] = df_diar["end"].astype(float) - df_diar["start"].astype(float)

print("File:", f"{RUN_DIR}/diarization.json")
print("Total segments:", len(df_diar))
print("Speakers:", sorted(df_diar["speaker"].unique()) if len(df_diar) else [])
print("Duration median:", df_diar["dur"].median() if len(df_diar) else 0)
print("Duration mean:", df_diar["dur"].mean() if len(df_diar) else 0)
print("< 1s:", int((df_diar["dur"] < 1).sum()) if len(df_diar) else 0)
print("< 2s:", int((df_diar["dur"] < 2).sum()) if len(df_diar) else 0)
print("< 3s:", int((df_diar["dur"] < 3).sum()) if len(df_diar) else 0)

display(df_diar[["index", "start", "end", "dur", "speaker"]].head(80))

In [ ]:
from pydub import AudioSegment
from IPython.display import Audio, display

full_audio = AudioSegment.from_file(AUDIO_WAV)

def listen_diar_segment(i, pad=0.2):
    s = diar_segments[i]
    start = max(0, float(s["start"]) - pad)
    end = float(s["end"]) + pad
    out = f"/kaggle/working/diar_review_{i:05d}.wav"
    full_audio[int(start * 1000):int(end * 1000)].export(out, format="wav")
    print(f'[{s["start"]:.2f} - {s["end"]:.2f}] {s["speaker"]}')
    display(Audio(out))

for i in range(min(10, len(diar_segments))):
    listen_diar_segment(i)

## 9. Stage 02 - Music/background clean

Output:
- `/kaggle/working/run_full/cleaned_audio.wav`
- `/kaggle/working/run_full/segment_flags.json`

In [ ]:
DEMUCS_ARG = "--demucs" if RUN_DEMUCS else "--no-demucs"

!python stage_02_music_clean.py \
  --input_audio "{AUDIO_WAV}" \
  --diarization_json "{RUN_DIR}/diarization.json" \
  --out_audio "{RUN_DIR}/cleaned_audio.wav" \
  --out_flags "{RUN_DIR}/segment_flags.json" \
  {DEMUCS_ARG}

In [ ]:
with open(f"{RUN_DIR}/segment_flags.json", "r", encoding="utf-8") as f:
    flags_data = json.load(f)

flags = flags_data.get("segment_demucs_flags", [])
print("Cleaned audio:", flags_data["audio_path"])
print("Segments:", len(flags_data["segments"]))
print("Demucs flagged segments:", sum(bool(x) for x in flags), "/", len(flags))

display(Audio(f"{RUN_DIR}/cleaned_audio.wav"))

## 10. Stage 03 - Overlap separation

Output:
- `/kaggle/working/run_full/segments.json`
- `/kaggle/working/run_full/separated_segments/*.wav` nếu có đoạn tách overlap.

In [ ]:
SEPREFORMER_ARG = "--sepreformer" if RUN_SEPREFORMER else "--no-sepreformer"

!python stage_03_overlap_separate.py \
  --cleaned_audio "{RUN_DIR}/cleaned_audio.wav" \
  --segment_flags_json "{RUN_DIR}/segment_flags.json" \
  --out_segments "{RUN_DIR}/segments.json" \
  --separated_dir "{RUN_DIR}/separated_segments" \
  {SEPREFORMER_ARG} \
  --sepreformer_path /kaggle/working/sommelier/SepReformer \
  --overlap_threshold 0.2

In [ ]:
with open(f"{RUN_DIR}/segments.json", "r", encoding="utf-8") as f:
    seg_data = json.load(f)

segments = seg_data["segments"]
df_seg = pd.DataFrame(segments)
df_seg["dur"] = df_seg["end"].astype(float) - df_seg["start"].astype(float)

print("Segments:", len(df_seg))
print("Separated:", int(df_seg.get("is_separated", pd.Series(dtype=bool)).fillna(False).sum()) if len(df_seg) else 0)
print("Duration median:", df_seg["dur"].median() if len(df_seg) else 0)
display(df_seg[[c for c in ["index", "start", "end", "dur", "speaker", "is_separated", "enhanced_audio_path"] if c in df_seg.columns]].head(80))

## 11. Cài cuDNN 8 riêng cho faster-whisper/ctranslate2

Không cài đè vào global torch. Chỉ cài vào `/kaggle/working/cudnn8` rồi truyền `LD_LIBRARY_PATH` khi chạy ASR.

In [ ]:
!rm -rf /kaggle/working/cudnn8
!python -m pip install --target /kaggle/working/cudnn8 nvidia-cudnn-cu12==8.9.7.29
!find /kaggle/working/cudnn8 -name "libcudnn_ops_infer.so.8"

## 12. Stage 04 - ASRMoE tiếng Việt trên 2 GPU

Output: `/kaggle/working/run_full/transcript.json`.

Cell này chạy cả 3 model ASR tiếng Việt. Whisper/faster-whisper chạy trên GPU `WHISPER_DEVICE_INDEX`; PhoWhisper và ChunkFormer chạy trên GPU `VI_ASR_DEVICE_INDEX`. Nếu vẫn OOM trên 2xT4, đổi `ASR_MOE = False` ở cell config để quay lại Whisper-only.

In [ ]:
!nvidia-smi

!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True LD_LIBRARY_PATH="/kaggle/working/cudnn8/nvidia/cudnn/lib:/kaggle/working/cudnn8/nvidia/cublas/lib:/kaggle/working/cudnn8/nvidia/cuda_nvrtc/lib:$LD_LIBRARY_PATH" \
python stage_04_asr.py \
  --segments_json "/kaggle/working/run_full/segments.json" \
  --audio "/kaggle/working/run_full/cleaned_audio.wav" \
  --out "/kaggle/working/run_full/transcript.json" \
  --ASRMoE \
  --no-whisperx_word_timestamps \
  --no-initprompt \
  --whisper_arch "large-v3" \
  --compute_type "float16" \
  --threads 4 \
  --whisper_device_index 0 \
  --vi_asr_device_index {VI_ASR_DEVICE_INDEX}

In [ ]:
with open(f"{RUN_DIR}/transcript.json", "r", encoding="utf-8") as f:
    transcript = json.load(f)

tr_segments = transcript["segments"]
print("Transcript segments:", len(tr_segments))
print("Metadata:", transcript.get("metadata", {}))

for s in tr_segments[:120]:
    start = float(s.get("start", 0))
    end = float(s.get("end", 0))
    speaker = s.get("speaker", "UNKNOWN")
    text = s.get("text", "").strip()
    print(f"[{start:07.2f} - {end:07.2f}] {speaker}: {text}")

## 13. Stage 05 - Export final JSON và audio segment MP3

Output:
- `/kaggle/working/run_full/final/data_audio.json`
- `/kaggle/working/run_full/final/data_audio/*.mp3`

In [ ]:
!python stage_05_export.py \
  --transcript_json "{RUN_DIR}/transcript.json" \
  --audio "{RUN_DIR}/cleaned_audio.wav" \
  --out_dir "{RUN_DIR}/final" \
  --audio_name data_audio

!find "{RUN_DIR}" -maxdepth 3 -type f | sort | head -200

## 14. Review full audio + full transcript

Cell này hiển thị full cleaned audio và toàn bộ transcript có speaker label trong một khung cuộn.

In [ ]:
from pathlib import Path
from IPython.display import HTML, Audio, display

display(Audio(f"{RUN_DIR}/cleaned_audio.wav"))

with open(f"{RUN_DIR}/transcript.json", "r", encoding="utf-8") as f:
    transcript = json.load(f)

lines = []
for s in transcript["segments"]:
    text = s.get("text", "").strip()
    if not text:
        continue
    start = float(s.get("start", 0))
    end = float(s.get("end", 0))
    speaker = s.get("speaker", "UNKNOWN")
    lines.append(f"[{start:07.2f} - {end:07.2f}] {speaker}: {text}")

full_script = "\n".join(lines)
out_txt = Path(RUN_DIR) / "final" / "full_transcript_with_speakers.txt"
out_txt.parent.mkdir(parents=True, exist_ok=True)
out_txt.write_text(full_script, encoding="utf-8")

print("Saved:", out_txt)
print("Lines:", len(lines))

safe = full_script.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
display(HTML(f"""
<div style="max-height: 720px; overflow-y: scroll; white-space: pre-wrap; font-family: ui-monospace, Menlo, Consolas, monospace; font-size: 14px; line-height: 1.5; border: 1px solid #ddd; padding: 12px; background: #fafafa;">
{safe}
</div>
"""))

## 15. Nghe từng đoạn export sau Stage 05

Dùng hàm `listen_exported_segment(i)` để nghe đoạn MP3 theo index trong thư mục final.

In [ ]:
from pathlib import Path
from IPython.display import Audio, display

FINAL_AUDIO_DIR = Path(RUN_DIR) / "final" / "data_audio"
mp3_files = sorted(FINAL_AUDIO_DIR.glob("*.mp3"))
print("MP3 segments:", len(mp3_files))
for p in mp3_files[:20]:
    print(p.name)

def listen_exported_segment(i):
    p = mp3_files[i]
    print(p)
    display(Audio(str(p)))

if mp3_files:
    listen_exported_segment(0)